# Challenge Lab: Module 07 -- Topic Modeling with LDA

**Course:** AI 102: Natural Language-Based Programming Techniques

- **Name:** Reezy Hudson
- **NetID:** rhudso10
- **Section:** 002

---

## Submission Guidelines

### What to Submit:
- This completed notebook with all cells run (use Runtime → Restart and run all)
- All `# TODO:` sections completed with working code
- All reflection questions answered in markdown

### File Naming:
- `LastName_FirstName_M07_TopicModeling.ipynb`

---

## Grading Rubric (100 Points)

| Part | Topic | Points |
|:-----|:------|-------:|
| Part 1 | Setup and Data Loading | -- |
| Part 2 | Text Preprocessing (TODOs 1-2) | 15 pts |
| Part 3 | gensim Dictionary and Corpus (TODOs 3-4) | 15 pts |
| Part 4 | Train the LDA Model (TODO 5) | 10 pts |
| Part 5 | Interpret Topics (TODOs 6-7) | 20 pts |
| Part 6 | Topic Distribution per Document (TODOs 8-9) | 10 pts |
| Part 7 | Coherence Score Comparison (TODO 10) | 15 pts |
| Part 8 | pyLDAvis (TODO 11) | 10 pts |
| Part 9 | Reflection Questions | 5 pts |
| | **Total** | **100 pts** |


## Learning Goals

By the end of this lab, you will be able to:

- Preprocess a real text corpus (CMU Book Summaries from Hugging Face) using `gensim.utils.simple_preprocess`
- Build a gensim **Dictionary and Corpus** from tokenized documents
- Train a **gensim LDA model** to discover hidden topics
- Interpret topics by reading their top keywords and assigning labels
- Use **coherence scores** to compare different topic counts objectively
- Visualize topics interactively using **pyLDAvis**

**Where to look for help:** the in-class workbook and the Module 07 guided lab use the exact same
gensim pipeline. Refer to them whenever you forget syntax.


## Part 1: Setup and Data Loading (provided)

We will use the **CMU Book Summary Dataset** from Hugging Face, which contains **16,559 book plot summaries** scraped from Wikipedia. Each row has the book's **title**, **author**, and **publication info**.

We will use this data to find hidden themes (topics) using **gensim LDA** -- the same library you used in
the in-class workbook and the guided lab.

**Run the next two cells exactly as they are.** They install libraries and load the data into a DataFrame.


In [ ]:
# ---- Run this cell as-is ----

# Install gensim 4.3+ and Hugging Face datasets
!pip install -q 'gensim>=4.3' pyLDAvis datasets

import nltk
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pprint import pprint
from datasets import load_dataset

# Download NLTK stopwords for filtering
nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords

import warnings
warnings.filterwarnings('ignore')

print('Setup complete!')


In [ ]:
# ---- Run this cell as-is ----

# Load all 16,559 book plot summaries from Hugging Face
raw_dataset = load_dataset('textminr/cmu-book-summaries', split='train')

# Convert to pandas DataFrame for easier handling
df = raw_dataset.to_pandas()

# Rename 'summary' column to 'text' for consistency with the rest of the lab
df = df.rename(columns={'summary': 'text'})

# Drop any rows with missing summaries
df = df.dropna(subset=['text']).reset_index(drop=True)

print(f'Total books: {len(df)}')
print(f'\nSample of titles in the dataset:')
print(df['title'].head(5).tolist())
print(f'\nFirst book:')
print(f'  Title: {df["title"].iloc[0]}')
print(f'  Author: {df["author"].iloc[0]}')
print(f'  Summary (first 300 chars): {df["text"].iloc[0][:300]}')


👀 **Expected:** ~16,559 books loaded with the first 5 titles shown and a preview of the first book's title, author, and summary. Note: dataset download takes 30-60 seconds the first time.


## Part 2: Text Preprocessing (TODOs 1-2) -- 15 pts

Before training a topic model, you need to clean each book summary. We use **`gensim.utils.simple_preprocess`**
(the same function from the guided lab), which lowercases the text, removes punctuation and numbers,
and returns a list of tokens. Then you remove stopwords so they don't dominate the topics.


### TODO 1: Tokenize all book summaries (8 pts)

**Goal:** Apply `simple_preprocess` (with `deacc=True`) to every book summary in `df['text']` and store the
result in a new column called `df['tokens']`. Each entry in the new column should be a list of clean tokens.

**Hint:** You will need to import `simple_preprocess` from `gensim.utils`. Use a `for` loop -- not a list comprehension.

deacc (bool, optional) – Remove accentuation if True.


In [ ]:
# ---- TODO 1: Tokenize each book summary with simple_preprocess ----
from gensim.utils import simple_preprocess

tokens_list = []
for text in df["text"]:
    tokens_list.append(simple_preprocess(text, deacc=True))

df["tokens"] = tokens_list


# ---- End TODO 1 ----

# Verify your work
print(f'Number of tokenized summaries: {len(df)}')
print(f'\nFirst 30 tokens of Book 0:')
print(df['tokens'].iloc[0][:30])


👀 **Expected:** A list of 30 lowercase tokens with no punctuation or numbers, e.g.

`['old', 'major', 'the', 'old', 'boar', 'on', 'the', 'manor', 'farm', 'calls', ...]`


### TODO 2: Remove stopwords from each tokenized book summary (7 pts)

**Goal:** Build a stopword list and apply it to every tokenized book summary. Store the result in a list called `data_words`.

#### Why we need MORE than NLTK stopwords

NLTK's default English stopwords (`the`, `and`, `is`, `of`, ...) only catch the most basic filler words. Book plot summaries contain many MORE filler words that aren't on NLTK's list but appear in every summary:

- `book`, `novel`, `story`, `chapter`, `narrator`, `protagonist`, `character`, `tale`
- `man`, `woman`, `boy`, `girl`, `young`, `family`, `life`, `time`, `home`
- `say`, `tell`, `find`, `come`, `take`, `make`, `know`, `think`

If we don't remove these, LDA will produce **garbage topics** where the same generic words show up in every topic. To save you the trouble of guessing which words to add, **we provide the full list below.**

**Requirements:**
1. Start with NLTK's English stopwords
2. Extend it with the `heavy_noise` list provided below
3. Use a `for` loop (not a list comprehension) to filter each summary's tokens
4. Store the result in `data_words`

**Hint -- copy this list into your code:**

```python
heavy_noise = [
    # Book/literary words
    'book', 'books', 'novel', 'novels', 'story', 'stories',
    'chapter', 'chapters', 'tale', 'tales', 'narrator', 'protagonist',
    'character', 'characters', 'plot', 'author', 'reader', 'readers',
    # Generic action verbs
    'go', 'goes', 'going', 'gone', 'went',
    'come', 'comes', 'coming', 'came',
    'get', 'gets', 'getting', 'got',
    'take', 'takes', 'taking', 'took', 'taken',
    'make', 'makes', 'making', 'made',
    'see', 'sees', 'seen', 'saw',
    'tell', 'tells', 'told', 'say', 'says', 'said',
    'find', 'finds', 'found', 'know', 'knows', 'knew',
    'try', 'tries', 'tried', 'leave', 'leaves', 'left',
    'use', 'uses', 'used', 'help', 'helps', 'helped',
    'think', 'thinks', 'thought', 'feel', 'feels', 'felt',
    # Generic people references
    'man', 'men', 'woman', 'women', 'people', 'person',
    'boy', 'girl', 'one', 'two', 'three', 'first', 'second',
    # Time and place filler
    'time', 'times', 'day', 'days', 'night', 'year', 'years',
    'home', 'house', 'place', 'way', 'world', 'life',
    # Vague modifiers
    'back', 'next', 'later', 'soon', 'still', 'now', 'then',
    'also', 'even', 'just', 'much', 'many', 'really',
    'would', 'could', 'might', 'may',
    # Generic descriptors
    'new', 'old', 'young', 'good', 'bad', 'big', 'little',
    'thing', 'things', 'something', 'anything', 'nothing',
]
```


In [ ]:
# ---- TODO 2: Build stopword list and remove stopwords from each summary ----

# Step 1: Load NLTK English stopwords
stop_words = stopwords.words('english')

heavy_noise = [
    # Book/literary words
    'book', 'books', 'novel', 'novels', 'story', 'stories',
    'chapter', 'chapters', 'tale', 'tales', 'narrator', 'protagonist',
    'character', 'characters', 'plot', 'author', 'reader', 'readers',
    # Generic action verbs
    'go', 'goes', 'going', 'gone', 'went',
    'come', 'comes', 'coming', 'came',
    'get', 'gets', 'getting', 'got',
    'take', 'takes', 'taking', 'took', 'taken',
    'make', 'makes', 'making', 'made',
    'see', 'sees', 'seen', 'saw',
    'tell', 'tells', 'told', 'say', 'says', 'said',
    'find', 'finds', 'found', 'know', 'knows', 'knew',
    'try', 'tries', 'tried', 'leave', 'leaves', 'left',
    'use', 'uses', 'used', 'help', 'helps', 'helped',
    'think', 'thinks', 'thought', 'feel', 'feels', 'felt',
    # Generic people references
    'man', 'men', 'woman', 'women', 'people', 'person',
    'boy', 'girl', 'one', 'two', 'three', 'first', 'second',
    # Time and place filler
    'time', 'times', 'day', 'days', 'night', 'year', 'years',
    'home', 'house', 'place', 'way', 'world', 'life',
    # Vague modifiers
    'back', 'next', 'later', 'soon', 'still', 'now', 'then',
    'also', 'even', 'just', 'much', 'many', 'really',
    'would', 'could', 'might', 'may',
    # Generic descriptors
    'new', 'old', 'young', 'good', 'bad', 'big', 'little',
    'thing', 'things', 'something', 'anything', 'nothing',
]

stop_words.extend(heavy_noise)
stop_words = list(set(stop_words))

# Remove duplicates (provided)
stop_words = list(set(stop_words))
print(f'Total stopwords: {len(stop_words)}')

data_words = []
for token_list in df['tokens']:
    filtered_tokens = []
    for word in token_list:
        # Add word only if it is NOT a stopword
        if word not in stop_words:
            filtered_tokens.append(word)
    data_words.append(filtered_tokens)


# Verify
print(f'Number of cleaned summaries: {len(data_words)}')
print(f'\nFirst 30 tokens of Book 0 after stopword removal:')
print(data_words[0][:30])


👀 **Expected:** Book 0 should contain only **content words** -- specific names, places, and plot details. No "the", "and", "story", "character", or any of the filler words from `heavy_noise`.

`First 30 tokens of Book 0 after stopword removal:
['major', 'boar', 'manor', 'farm', 'calls', 'animals', 'farm', 'meeting', 'compares', 'humans', 'parasites', 'teaches', 'animals', 'revolutionary', 'song', 'beasts', 'england', 'major', ...]`


## Part 3: Build the gensim Dictionary and Corpus (TODOs 3-4) -- 15 pts

Now convert the cleaned text into the data structures gensim needs:

- **Dictionary** -- maps every unique word to an integer ID
- **Corpus** -- each document becomes a list of `(word_id, count)` tuples (bag-of-words)

**This is the same pipeline you used in the guided lab.** Refer back to it for syntax.


### TODO 3: Build the gensim Dictionary (5 pts)

**Goal:** Build a gensim `Dictionary` from `data_words` and store it in a variable called `id2word`.
Then filter out very rare and very common words to keep the vocabulary manageable.

**Hint:** Look up `corpora.Dictionary` in the guided lab. Use `id2word.filter_extremes(no_below=5, no_above=0.5)`
to keep only words that appear in at least 5 summaries and at most 50% of all summaries.


In [ ]:
# ---- TODO 3: Build the Dictionary and filter extremes ----
from gensim import corpora

id2word = corpora.Dictionary(data_words)
id2word.filter_extremes(no_below=5, no_above=0.5)


# ---- End TODO 3 ----

# Verify
print(f'Vocabulary size after filtering: {len(id2word)}')
print(f'\nSample words and their IDs:')
for word_id in list(id2word.token2id.values())[:10]:
    print(f'  ID {word_id}: {id2word[word_id]}')


👀 **Expected:** Vocabulary size should be in the **5,000-15,000** range after filtering.

`Vocabulary size after filtering: 31087`


### TODO 4: Build the bag-of-words Corpus (10 pts)

**Goal:** Convert every document in `data_words` into a bag-of-words representation using
`id2word.doc2bow()`. Store the result in a list called `bow_corpus`.

**Hint:** Use a `for` loop (not a list comprehension). Each entry in `bow_corpus` should be a list of
`(word_id, count)` tuples.


In [ ]:
# ---- TODO 4: Build the bag-of-words corpus ----

bow_corpus = []

for doc in data_words:
    bow_corpus.append(id2word.doc2bow(doc))


# ---- End TODO 4 ----

# Verify
print(f'Number of documents in corpus: {len(bow_corpus)}')
print(f'\nFirst 10 (word_id, count) pairs of Document 0:')
print(bow_corpus[0][:10])

print(f'\nSame pairs as (word, count) for readability:')
for word_id, count in bow_corpus[0][:10]:
    print(f'  {id2word[word_id]}: {count}')


👀 **Expected:** A list of `(word_id, count)` tuples for Document 0, then the same data shown
as `(word, count)` pairs.


## Part 4: Train the LDA Model (TODO 5) -- 10 pts

Now train LDA on your bag-of-words corpus. Start with **5 topics** as a reasonable starting point.
You will experiment with different topic counts in Part 7.


### TODO 5: Train the gensim LdaModel (10 pts)

**Goal:** Train an LDA model with **5 topics** and store it in a variable called `lda_model`.

**Requirements:**
- Use `gensim.models.LdaMulticore` - multicore version for better performance
- Pass your `bow_corpus` and `id2word` from the previous step
- Use `random_state=42` for reproducibility
- Use `passes=10` for training iterations
- Use `workers=2` (Colab free tier has 2 vCPUs. Colab Pro has 4.)

Refer to the guided lab if you forget the parameter names.

**⚠️ MAJOR PERFORMANCE WARNING:**
- **Expected total runtime: 3-5 minutes.**
- **Use `workers=4` if on Colab Pro**


In [ ]:
# ---- TODO 5: Train the LDA model ----
from gensim.models import LdaMulticore

num_topics = 5

lda_model = LdaMulticore(
    corpus=bow_corpus,
    id2word=id2word,
    num_topics=num_topics,
    random_state=42,
    passes=10,
    workers=2
)


# ---- End TODO 5 ----

# Verify
print(f'LDA model trained with {lda_model.num_topics} topics!')


👀 **Expected:** "LDA model trained with 5 topics!". Training takes 1-2 minutes.


## Part 5: Interpret Topics (TODOs 6-7) -- 20 pts

**This is the most important step.** LDA gives you word groups, not labels. **You** read the keywords
and decide what each topic is about.


### TODO 6: Print the top 10 words for each topic (10 pts)

**Goal:** Loop over all 5 topics and print the top 10 keywords for each. Format the output cleanly so
you can read it.

**Hint:** Use `lda_model.show_topic(topic_id, topn=10)`. It returns a list of `(word, weight)` tuples.


In [ ]:
# ---- TODO 6: Print the top 10 words for each topic ----

for topic_id in range(num_topics):
    print(f"\nTopic {topic_id}")
    print("-" * 40)

    words = lda_model.show_topic(topic_id, topn=10)
    for word, weight in words:
        print(f"{word:<15} {weight:.4f}")


# ---- End TODO 6 ----


👀 **Expected:** 5 topics, each with their top 10 keywords. Look for patterns -- some topics will
clearly relate to literary genres like fantasy, sci-fi, religious texts, war stories, romance, etc.


### TODO 7: Assign a label to each topic (5 pts)

Look at the top words you printed and **give each topic a short descriptive name** (2-4 words).
Fill in the dictionary below.

**Examples of good labels:** `'action and violence'`, `'romantic drama'`, `'sci-fi and aliens'`, `'crime thriller'`


In [ ]:
# ---- TODO 7: Assign topic labels ----

topic_labels = {
    0: 'LABEL_HERE',
    1: 'LABEL_HERE',
    2: 'LABEL_HERE',
    3: 'LABEL_HERE',
    4: 'LABEL_HERE',
}

for topic_id, label in topic_labels.items():
    print(f'Topic {topic_id}: {label}')


👀 **Expected:** Your 5 topic labels printed to the screen. Each label should be a 2-4 word phrase that captures the theme of that topic's top keywords.
